[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/a-ramakrishnan/vision_transformers/blob/main/ddpm.ipynb)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
print(f"Using device: {device}")

In [ ]:
BATCH_SIZE = 128

# Scale MNIST images from [0, 1] to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: (x * 2.0) - 1.0),
])

train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

In [ ]:
# =====================================================================
# PART 1: Linear Noise Scheduler (Forward & Reverse Transition Kernels)
# =====================================================================
class LinearNoiseScheduler:
    def __init__(self, num_timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cpu"):
        self.num_timesteps = num_timesteps
        self.beta_start = beta_start
        self.beta_end = beta_end

        self.betas = torch.linspace(beta_start, beta_end, num_timesteps, device=device)
        self.alphas = 1.0 - self.betas
        self.alpha_cum_prod = torch.cumprod(self.alphas, dim=0)
        self.sqrt_alpha_cum_prod = torch.sqrt(self.alpha_cum_prod)
        self.sqrt_one_minus_alpha_cum_prod = torch.sqrt(1.0 - self.alpha_cum_prod)

    def add_noise(self, original, noise, t):
        # Forward transition kernel: q(x_t | x_0) = sqrt(alpha_bar_t) * x_0 + sqrt(1 - alpha_bar_t) * eps
        sqrt_alpha_cum_prod = self.sqrt_alpha_cum_prod[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_cum_prod = self.sqrt_one_minus_alpha_cum_prod[t].view(-1, 1, 1, 1)
        return sqrt_alpha_cum_prod * original + sqrt_one_minus_alpha_cum_prod * noise

    def sample_prev_timestep(self, xt, noise_pred, t):
        # Reverse transition kernel: p_theta(x_{t-1} | x_t)
        x0 = (xt - (self.sqrt_one_minus_alpha_cum_prod[t] * noise_pred)) / self.sqrt_alpha_cum_prod[t]
        x0 = torch.clamp(x0, -1.0, 1.0)

        mean = xt - ((self.betas[t] * noise_pred) / self.sqrt_one_minus_alpha_cum_prod[t])
        mean = mean / torch.sqrt(self.alphas[t])

        if t == 0:
            return mean, x0
        else:
            variance = (1.0 - self.alpha_cum_prod[t - 1]) / (1.0 - self.alpha_cum_prod[t])
            variance = variance * self.betas[t]
            sigma = variance ** 0.5
            z = torch.randn_like(xt)
            return mean + sigma * z, x0

In [ ]:
scheduler = LinearNoiseScheduler(num_timesteps=1000, beta_start=1e-4, beta_end=0.02, device=device)

sample_imgs, _ = next(iter(train_loader))
x0 = sample_imgs[0:1].to(device)
timesteps_to_show = [0, 100, 300, 600, 999]

fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(10, 2.5))
for idx, step in enumerate(timesteps_to_show):
    t_tensor = torch.tensor([step], device=device)
    noise = torch.randn_like(x0)
    xt = scheduler.add_noise(x0, noise, t_tensor)
    img_display = (xt.cpu().squeeze() + 1.0) / 2.0
    axes[idx].imshow(img_display.clamp(0.0, 1.0), cmap="gray")
    axes[idx].set_title(f"t = {step}")
    axes[idx].axis("off")
plt.suptitle("Forward Diffusion Process q(x_t | x_0)")
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# PART 2: Sinusoidal Time Embeddings & Time-Conditioned U-Net
# =====================================================================
def get_time_embedding(time_steps, temb_dim):
    half_dim = temb_dim // 2
    factor = 10000 ** (
        torch.arange(start=0, end=half_dim, dtype=torch.float32, device=time_steps.device) / half_dim
    )
    t_emb = time_steps[:, None].repeat(1, half_dim) / factor
    t_emb = torch.cat([torch.sin(t_emb), torch.cos(t_emb)], dim=-1)
    return t_emb


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, down_sample=True, num_heads=4):
        super().__init__()
        self.down_sample = down_sample
        self.resnet_conv_first = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
        )
        self.t_emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels),
        )
        self.resnet_conv_second = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
        )
        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.down_sample_conv = (
            nn.Conv2d(out_channels, out_channels, kernel_size=4, stride=2, padding=1)
            if self.down_sample
            else nn.Identity()
        )

    def forward(self, x, t_emb):
        out = x
        resnet_input = out
        out = self.resnet_conv_first(out)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out)
        out = out + self.residual_input_conv(resnet_input)

        batch_size, channels, h, w = out.shape
        in_attn = out.reshape(batch_size, channels, h * w)
        in_attn = self.attention_norm(in_attn).transpose(1, 2)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
        out = out + out_attn

        out = self.down_sample_conv(out)
        return out


class MidBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, num_heads=4):
        super().__init__()
        self.resnet_conv_first = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, in_channels),
                nn.SiLU(),
                nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
            ),
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            ),
        ])
        self.t_emb_layers = nn.ModuleList([
            nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels)),
            nn.Sequential(nn.SiLU(), nn.Linear(t_emb_dim, out_channels)),
        ])
        self.resnet_conv_second = nn.ModuleList([
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            ),
            nn.Sequential(
                nn.GroupNorm(8, out_channels),
                nn.SiLU(),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            ),
        ])
        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.ModuleList([
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
        ])

    def forward(self, x, t_emb):
        out = x
        resnet_input = out
        out = self.resnet_conv_first[0](out)
        out = out + self.t_emb_layers[0](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[0](out)
        out = out + self.residual_input_conv[0](resnet_input)

        batch_size, channels, h, w = out.shape
        in_attn = out.reshape(batch_size, channels, h * w)
        in_attn = self.attention_norm(in_attn).transpose(1, 2)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
        out = out + out_attn

        resnet_input = out
        out = self.resnet_conv_first[1](out)
        out = out + self.t_emb_layers[1](t_emb)[:, :, None, None]
        out = self.resnet_conv_second[1](out)
        out = out + self.residual_input_conv[1](resnet_input)
        return out


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, t_emb_dim, up_sample=True, num_heads=4):
        super().__init__()
        self.up_sample = up_sample
        self.resnet_conv_first = nn.Sequential(
            nn.GroupNorm(8, in_channels),
            nn.SiLU(),
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1),
        )
        self.t_emb_layers = nn.Sequential(
            nn.SiLU(),
            nn.Linear(t_emb_dim, out_channels),
        )
        self.resnet_conv_second = nn.Sequential(
            nn.GroupNorm(8, out_channels),
            nn.SiLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
        )
        self.attention_norm = nn.GroupNorm(8, out_channels)
        self.attention = nn.MultiheadAttention(out_channels, num_heads, batch_first=True)
        self.residual_input_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.up_sample_conv = (
            nn.ConvTranspose2d(in_channels // 2, in_channels // 2, kernel_size=4, stride=2, padding=1)
            if self.up_sample
            else nn.Identity()
        )

    def forward(self, x, out_down, t_emb):
        x = self.up_sample_conv(x)
        x = torch.cat([x, out_down], dim=1)

        out = x
        resnet_input = out
        out = self.resnet_conv_first(out)
        out = out + self.t_emb_layers(t_emb)[:, :, None, None]
        out = self.resnet_conv_second(out)
        out = out + self.residual_input_conv(resnet_input)

        batch_size, channels, h, w = out.shape
        in_attn = out.reshape(batch_size, channels, h * w)
        in_attn = self.attention_norm(in_attn).transpose(1, 2)
        out_attn, _ = self.attention(in_attn, in_attn, in_attn)
        out_attn = out_attn.transpose(1, 2).reshape(batch_size, channels, h, w)
        out = out + out_attn
        return out


class UNet(nn.Module):
    def __init__(self, im_channels=1):
        super().__init__()
        self.down_channels = [32, 64, 128, 256]
        self.mid_channels = [256, 256, 128]
        self.t_emb_dim = 128
        self.down_sample = [True, True, False]

        self.t_proj = nn.Sequential(
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
            nn.SiLU(),
            nn.Linear(self.t_emb_dim, self.t_emb_dim),
        )
        self.up_sample = list(reversed(self.down_sample))
        self.conv_in = nn.Conv2d(im_channels, self.down_channels[0], kernel_size=3, padding=1)

        self.downs = nn.ModuleList([
            DownBlock(
                self.down_channels[i],
                self.down_channels[i + 1],
                self.t_emb_dim,
                down_sample=self.down_sample[i],
                num_heads=4,
            )
            for i in range(len(self.down_channels) - 1)
        ])

        self.mids = nn.ModuleList([
            MidBlock(
                self.mid_channels[i],
                self.mid_channels[i + 1],
                self.t_emb_dim,
                num_heads=4,
            )
            for i in range(len(self.mid_channels) - 1)
        ])

        self.ups = nn.ModuleList([
            UpBlock(
                self.down_channels[i] * 2,
                self.down_channels[i - 1] if i != 0 else 16,
                self.t_emb_dim,
                up_sample=self.down_sample[i],
                num_heads=4,
            )
            for i in reversed(range(len(self.down_channels) - 1))
        ])

        self.norm_out = nn.GroupNorm(8, 16)
        self.conv_out = nn.Conv2d(16, im_channels, kernel_size=3, padding=1)

    def forward(self, x, t):
        out = self.conv_in(x)
        t_emb = get_time_embedding(torch.as_tensor(t).long(), self.t_emb_dim)
        t_emb = self.t_proj(t_emb)

        down_outs = []
        for down in self.downs:
            down_outs.append(out)
            out = down(out, t_emb)

        for mid in self.mids:
            out = mid(out, t_emb)

        for up in self.ups:
            down_out = down_outs.pop()
            out = up(out, down_out, t_emb)

        out = self.norm_out(out)
        out = F.silu(out)
        out = self.conv_out(out)
        return out

In [ ]:
# =====================================================================
# PART 3: DDPM Training Loop
# =====================================================================
NUM_TIMESTEPS = 1000
NUM_EPOCHS = 20
LR = 1e-4

model = UNet(im_channels=1).to(device)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()

for epoch in range(NUM_EPOCHS):
    model.train()
    losses = []
    for images, _ in train_loader:
        images = images.to(device)
        noise = torch.randn_like(images)
        t = torch.randint(0, NUM_TIMESTEPS, (images.shape[0],), device=device)

        noisy_images = scheduler.add_noise(images, noise, t)
        noise_pred = model(noisy_images, t)

        loss = criterion(noise_pred, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    mean_loss = sum(losses) / len(losses)
    print(f"[DDPM] Epoch [{epoch + 1}/{NUM_EPOCHS}] - Noise MSE Loss: {mean_loss:.6f}")

torch.save(model.state_dict(), "ddpm_mnist.pth")

In [ ]:
# =====================================================================
# PART 4: Reverse Diffusion Sampling (Generation from Pure Noise)
# =====================================================================
@torch.no_grad()
def sample_ddpm(model, scheduler, num_samples=16, im_size=28, im_channels=1):
    model.eval()
    xt = torch.randn((num_samples, im_channels, im_size, im_size), device=device)

    for i in reversed(range(scheduler.num_timesteps)):
        t_batch = torch.full((num_samples,), i, device=device, dtype=torch.long)
        noise_pred = model(xt, t_batch)
        xt, _ = scheduler.sample_prev_timestep(xt, noise_pred, i)

    imgs = torch.clamp(xt, -1.0, 1.0).cpu()
    imgs = (imgs + 1.0) / 2.0
    return imgs


generated_imgs = sample_ddpm(model, scheduler, num_samples=16)

fig, axes = plt.subplots(2, 8, figsize=(12, 3))
for i in range(16):
    r, c = divmod(i, 8)
    axes[r, c].imshow(generated_imgs[i].squeeze(), cmap="gray")
    axes[r, c].axis("off")
plt.suptitle("DDPM: Generated MNIST Digits from Pure Gaussian Noise")
plt.tight_layout()
plt.show()